In [1]:
from pathlib import Path

import pandas as pd



In [2]:
INPUT_PATH = Path("data/processed/stint_laps_2024.parquet")


def main():
    df = pd.read_parquet(INPUT_PATH)

    print("=" * 70)
    print("INSPECCIÓN DE CALIDAD DE STINTS")
    print("=" * 70)

    # ---------------------------------------------------------
    # 1. Información general
    # ---------------------------------------------------------

    stint_cols = ["Race", "Driver", "Stint"]

    stints = (
        df.groupby(stint_cols)
        .agg(
            StintLaps=("StintLap", "count"),
            FirstLap=("LapNumber", "min"),
            LastLap=("LapNumber", "max"),
            FirstTyreLife=("TyreLife", "min"),
            LastTyreLife=("TyreLife", "max"),
            Compound=("Compound", "first"),
            Team=("Team", "first"),
        )
        .reset_index()
    )

    print()
    print("1. RESUMEN GENERAL")
    print("-" * 70)

    print(f"Vueltas: {len(df):,}")
    print(f"Stints: {len(stints):,}")
    print(f"Carreras: {df['Race'].nunique()}")
    print(f"Pilotos: {df['Driver'].nunique()}")
    print(f"Equipos: {df['Team'].nunique()}")

    # ---------------------------------------------------------
    # 2. Stints por compuesto
    # ---------------------------------------------------------

    print()
    print("2. STINTS POR COMPUESTO")
    print("-" * 70)

    print(
        stints["Compound"]
        .value_counts()
        .sort_index()
    )

    # ---------------------------------------------------------
    # 3. Duración de los stints
    # ---------------------------------------------------------

    print()
    print("3. DURACIÓN DE LOS STINTS")
    print("-" * 70)

    print(stints["StintLaps"].describe())

    # ---------------------------------------------------------
    # 4. Duración por compuesto
    # ---------------------------------------------------------

    print()
    print("4. DURACIÓN POR COMPUESTO")
    print("-" * 70)

    duration_by_compound = (
        stints
        .groupby("Compound")["StintLaps"]
        .agg(
            Count="count",
            Mean="mean",
            Median="median",
            Min="min",
            Max="max",
        )
        .round(2)
    )

    print(duration_by_compound)

    # ---------------------------------------------------------
    # 5. Stints cortos
    # ---------------------------------------------------------

    print()
    print("5. STINTS CORTOS")
    print("-" * 70)

    for threshold in [1, 2, 3, 5, 10]:
        count = (stints["StintLaps"] <= threshold).sum()
        percentage = count / len(stints) * 100

        print(
            f"Stints con <= {threshold:2d} vueltas: "
            f"{count:4d} ({percentage:5.1f}%)"
        )

    # ---------------------------------------------------------
    # 6. Stints suficientemente largos
    # ---------------------------------------------------------

    print()
    print("6. STINTS LARGOS")
    print("-" * 70)

    for threshold in [5, 10, 15, 20, 25, 30]:
        count = (stints["StintLaps"] >= threshold).sum()
        percentage = count / len(stints) * 100

        print(
            f"Stints con >= {threshold:2d} vueltas: "
            f"{count:4d} ({percentage:5.1f}%)"
        )

    # ---------------------------------------------------------
    # 7. TyreLife inicial
    # ---------------------------------------------------------

    print()
    print("7. TYRELIFE INICIAL")
    print("-" * 70)

    print(
        stints["FirstTyreLife"]
        .value_counts()
        .sort_index()
        .head(20)
    )

    print()
    print(
        "Stints que empiezan con TyreLife > 1:",
        (stints["FirstTyreLife"] > 1).sum(),
    )

    print(
        "Porcentaje:",
        round(
            (stints["FirstTyreLife"] > 1).mean() * 100,
            2,
        ),
        "%",
    )

    # ---------------------------------------------------------
    # 8. Diferencia entre StintLap y TyreLife
    # ---------------------------------------------------------

    df["TyreLifeMinusStintLap"] = (
        df["TyreLife"] - df["StintLap"]
    )

    print()
    print("8. RELACIÓN STINTLAP / TYRELIFE")
    print("-" * 70)

    print(
        df["TyreLifeMinusStintLap"]
        .describe()
        .round(2)
    )

    print()
    print("Distribución de las diferencias más frecuentes:")

    print(
        df["TyreLifeMinusStintLap"]
        .value_counts()
        .sort_index()
        .head(20)
    )

    # ---------------------------------------------------------
    # 9. Saltos dentro de los stints
    # ---------------------------------------------------------

    print()
    print("9. SALTOS DE LAPNUMBER DENTRO DE UN STINT")
    print("-" * 70)

    df = df.sort_values(
        ["Race", "Driver", "Stint", "StintLap"]
    ).copy()

    df["LapNumberDiff"] = (
        df.groupby(stint_cols)["LapNumber"]
        .diff()
    )

    gaps = df[df["LapNumberDiff"] > 1]

    print(f"Vueltas con salto: {len(gaps):,}")

    if len(gaps) > 0:
        print()
        print("Primeros ejemplos:")
        print(
            gaps[
                [
                    "Race",
                    "Driver",
                    "Stint",
                    "StintLap",
                    "LapNumber",
                    "LapNumberDiff",
                    "TyreLife",
                ]
            ]
            .head(20)
            .to_string(index=False)
        )

    # ---------------------------------------------------------
    # 10. Compuestos
    # ---------------------------------------------------------

    print()
    print("10. VUELTAS POR COMPUESTO")
    print("-" * 70)

    print(
        df["Compound"]
        .value_counts()
        .sort_index()
    )

    print()
    print("=" * 70)
    print("FIN DE LA INSPECCIÓN")
    print("=" * 70)


if __name__ == "__main__":
    main()

INSPECCIÓN DE CALIDAD DE STINTS

1. RESUMEN GENERAL
----------------------------------------------------------------------
Vueltas: 23,256
Stints: 1,179
Carreras: 24
Pilotos: 24
Equipos: 10

2. STINTS POR COMPUESTO
----------------------------------------------------------------------
Compound
HARD            493
INTERMEDIATE     98
MEDIUM          448
SOFT            137
WET               3
Name: count, dtype: int64

3. DURACIÓN DE LOS STINTS
----------------------------------------------------------------------
count    1179.000000
mean       19.725191
std        11.482287
min         1.000000
25%        11.000000
50%        19.000000
75%        26.000000
max        76.000000
Name: StintLaps, dtype: float64

4. DURACIÓN POR COMPUESTO
----------------------------------------------------------------------
              Count   Mean  Median  Min  Max
Compound                                    
HARD            493  24.97    24.0    2   76
INTERMEDIATE     98  18.13    20.5    1   37
MED